# Fase 2b: verificador de hoja de café, versión 2
Tesis: aplicación móvil para identificar síntomas de ojo de gallo (Rodrigo Alfaro, UCC).

**Por qué existe este cuaderno.** La revisión del 30 de octubre de 2026 encontró que el verificador de la Fase 2
acepta como "hoja de café" imágenes artificiales de un solo color (gris, negro o blanco). Este cuaderno:
1. reentrena el verificador con hojas de café de dos fuentes (Mendeley y BRACOL) y con más tipos de imágenes que no son hojas;
2. compara el verificador anterior (v1) con el nuevo (v2) en los mismos conjuntos de prueba;
3. calcula el umbral del filtro de calidad (desviación de los píxeles) con las hojas reales;
4. convierte el v2 a TensorFlow Lite para la aplicación.

**Cómo usarlo:** Entorno de ejecución → Cambiar tipo de entorno → GPU (T4). Luego Entorno de ejecución → Ejecutar todas.
Todo lo que se reporta en la tesis sale de las tablas que este cuaderno guarda en `fase2b_verificador/`.

In [1]:
import os, sys, json, math, time, random, glob, platform
MOCK = os.environ.get("MOCK") == "1"   # solo para probar el cuaderno fuera de Colab
if not MOCK:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, tensorflow as tf, keras
from PIL import Image, ImageOps, ImageFilter
from concurrent.futures import ThreadPoolExecutor

SEMILLA = 42
TAM = 224
UMBRAL_VERIFICADOR = 0.5
EPOCAS = int(os.environ.get("EPOCAS", 15))
N_NEG_TFDS = int(os.environ.get("N_NEG", 400))
BASE = os.environ.get("RUTA_BASE", "/content/drive/MyDrive/tesis_ojo_de_gallo")
DS = os.path.join(BASE, "02_Dataset")
RUTAS = {
    "ojo_de_gallo": os.path.join(DS, "Ojo_Gallo"),
    "sana": os.path.join(DS, "Sanas"),
    "negativos_propios": os.path.join(DS, "negativos_propios"),
    "bracol": os.path.join(DS, "externos", "BRACOL", "leaf", "images"),
    "pg26038": os.path.join(DS, "externos", "PG26038_muestras", "images"),
    "verificador_v1": os.path.join(BASE, "fase2_resultados", "modelos", "verificador_hoja_de_cafe.keras"),
}
SAL = os.path.join(BASE, "fase2b_verificador")
for sub in ["tablas", "modelos", "android", "figuras"]:
    os.makedirs(os.path.join(SAL, sub), exist_ok=True)
EXT = (".jpg", ".jpeg", ".png", ".bmp")

def fijar_semillas():
    random.seed(SEMILLA); np.random.seed(SEMILLA); keras.utils.set_random_seed(SEMILLA)
fijar_semillas()

VERSIONES = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
             "numpy": np.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
print(VERSIONES)
for k, r in RUTAS.items():
    print(("OK   " if os.path.exists(r) else "FALTA"), k, r)

def guardar_json(obj, nombre):
    with open(os.path.join(SAL, "tablas", nombre), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)

def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

Mounted at /content/drive
{'python': '3.13.16', 'tensorflow': '2.21.0', 'keras': '3.13.2', 'numpy': '2.1.3', 'gpu': ['/physical_device:GPU:0']}
OK    ojo_de_gallo /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Ojo_Gallo
OK    sana /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Sanas
OK    negativos_propios /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/negativos_propios
OK    bracol /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/BRACOL/leaf/images
OK    pg26038 /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/PG26038_muestras/images
OK    verificador_v1 /content/drive/MyDrive/tesis_ojo_de_gallo/fase2_resultados/modelos/verificador_hoja_de_cafe.keras


## 1. Lectura de imágenes
Todas las imágenes se leen igual que en la aplicación: orientación EXIF corregida, RGB (las PNG con fondo
transparente se componen sobre blanco), redimensión bilineal de la imagen completa a 224 × 224 con
`tf.image.resize` y valores de 0 a 255 en float32.

In [2]:
def leer(ruta):
    with Image.open(ruta) as im:
        im = ImageOps.exif_transpose(im)
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA")
            fondo = Image.new("RGBA", im.size, (255, 255, 255, 255))
            im = Image.alpha_composite(fondo, im)
        arr = np.asarray(im.convert("RGB"), dtype=np.float32)
    return tf.image.resize(arr[None], (TAM, TAM), method="bilinear").numpy()[0]

DANADAS = []
def leer_seguro(ruta):
    try:
        return leer(ruta)
    except Exception as e:
        DANADAS.append({"imagen": ruta, "error": str(e)}); return None

def leer_varias(rutas, desc=""):
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        X = [x for x in ex.map(leer_seguro, rutas) if x is not None]
    print(f"{desc}: {len(X)} imágenes leídas de {len(rutas)} en {time.time() - t0:.0f} s")
    return np.stack(X).astype("float32") if X else np.zeros((0, TAM, TAM, 3), "float32")

def listar(carpeta):
    return sorted(os.path.join(carpeta, f) for f in os.listdir(carpeta) if f.lower().endswith(EXT)) if os.path.isdir(carpeta) else []

def dividir(n, fracciones=(0.70, 0.15, 0.15), semilla=SEMILLA):
    idx = np.random.RandomState(semilla).permutation(n)
    a = int(round(fracciones[0] * n)); b = a + int(round(fracciones[1] * n))
    return {"train": idx[:a], "validation": idx[a:b], "test": idx[b:]}

## 2. Hojas de café (positivos)
- **Mendeley** (Santa Maria Pinedo, 2026): las 999 hojas con la misma partición de la Fase 2 (699 / 149 / 151).
- **BRACOL** (Krohling et al., 2019): 1.402 hojas de café arábica de Brasil, partición aleatoria 70/15/15 con semilla 42.
- **PG 26038** (Chery & Morales Peña, 2026): 100 imágenes de 50 hojas; **solo prueba**, ningún modelo las ve en el entrenamiento.

In [3]:
PARTICION = json.loads(r'''{"train": {"ojo_de_gallo": ["20251129_091959.jpg", "20251129_092016.jpg", "20251129_092053.jpg", "20251129_092106.jpg", "20251129_092123.jpg", "20251129_092139.jpg", "20251129_092149.jpg", "20251129_092158.jpg", "20251129_092220.jpg", "20251129_092252.jpg", "20251129_092438.jpg", "20251129_092457.jpg", "20251129_092504.jpg", "20251129_092511.jpg", "20251129_092539.jpg", "20251129_092559.jpg", "20251129_092702.jpg", "20251129_092708.jpg", "20251129_092800.jpg", "20251129_092931.jpg", "20251129_093055.jpg", "20251129_093132.jpg", "20251129_093207.jpg", "20251129_093218.jpg", "20251129_093257.jpg", "20251129_093304.jpg", "20251129_093324.jpg", "20251129_093359.jpg", "20251129_093502.jpg", "20251129_093615.jpg", "20251129_093623.jpg", "20251129_093725.jpg", "20251129_093729.jpg", "20251129_093736.jpg", "20251129_093956.jpg", "20251129_094006.jpg", "20251129_094032.jpg", "20251129_094102.jpg", "20251129_094107.jpg", "20251129_094130.jpg", "20251129_094219.jpg", "20251129_094307.jpg", "20251129_094337.jpg", "20251129_094614.jpg", "20251129_094622.jpg", "20251129_094627.jpg", "20251129_094654.jpg", "20251129_094701.jpg", "20251129_094834.jpg", "20251129_094950.jpg", "20251129_095011.jpg", "20251129_095106.jpg", "20251129_095132.jpg", "20251129_095138.jpg", "20251129_095206.jpg", "20251129_095244.jpg", "20251129_095331.jpg", "20251129_095408.jpg", "20251129_095416.jpg", "20251129_095449.jpg", "20251129_095512.jpg", "20251129_095538.jpg", "20251129_095552.jpg", "20251129_095556.jpg", "20251129_095601.jpg", "20251129_095609.jpg", "20251129_095629.jpg", "20251129_095642.jpg", "20251129_095712.jpg", "20251129_095729.jpg", "20251129_095736.jpg", "20251129_095755.jpg", "20251129_095810.jpg", "20251129_095821.jpg", "20251129_100046.jpg", "20251129_100416.jpg", "20251129_100436.jpg", "20251129_100506.jpg", "20251129_100515.jpg", "20251129_100535.jpg", "20251129_100547.jpg", "20251129_100621.jpg", "20251129_100711.jpg", "20251129_100808.jpg", "20251129_100906.jpg", "20251129_100918.jpg", "20251129_100948.jpg", "20251129_100955.jpg", "20251129_101023.jpg", "20251129_101029.jpg", "20251129_101033.jpg", "20251129_101157.jpg", "20251129_101255.jpg", "20251129_101307.jpg", "20251129_101319.jpg", "20251129_101337.jpg", "20251129_101342.jpg", "20251129_101430.jpg", "20251129_101441.jpg", "20251129_101504.jpg", "20251129_101509.jpg", "20251129_101518.jpg", "20251129_101529.jpg", "20251129_101559.jpg", "20251129_101610.jpg", "20251129_101624.jpg", "20251129_101641.jpg", "20251129_101714.jpg", "20251129_101831.jpg", "20251129_101839.jpg", "20251129_101856.jpg", "20251129_101905.jpg", "20251129_102019.jpg", "20251129_102125.jpg", "20251129_102402.jpg", "20251129_102518.jpg", "20251129_102657.jpg", "20251129_102737.jpg", "20251129_102838.jpg", "20251129_102905.jpg", "20251129_103003.jpg", "20251129_103051.jpg", "20251129_103124.jpg", "20251129_103139.jpg", "20251129_103154.jpg", "20251129_103206.jpg", "20251129_103304.jpg", "20251129_103311.jpg", "20251129_103353.jpg", "20251129_103409.jpg", "20251129_103437.jpg", "20251129_103445.jpg", "20251129_103532.jpg", "20251129_103542.jpg", "20251129_103604.jpg", "20251129_103618.jpg", "20251129_103647.jpg", "20251129_103728.jpg", "20251129_103758.jpg", "20251129_103804.jpg", "20251129_103839.jpg", "20251129_103847.jpg", "20251129_103938.jpg", "20251129_104212.jpg", "20251129_104314.jpg", "20251129_104320.jpg", "20251129_104346.jpg", "20251129_104351.jpg", "20251129_104356.jpg", "20251129_104409.jpg", "20251129_104557.jpg", "20251129_104706.jpg", "20251129_104727.jpg", "20251129_104745.jpg", "20251129_104803.jpg", "20251129_104901.jpg", "20251129_104924.jpg", "20251129_104958.jpg", "20251129_105257.jpg", "20251129_105404.jpg", "20251129_105435.jpg", "20251129_105505.jpg", "20251129_105517.jpg", "20251129_105537.jpg", "20251129_114546.jpg", "20251129_114556.jpg", "20251129_114630.jpg", "20251129_114655.jpg", "20251129_114830.jpg", "20251129_114858.jpg", "20251129_114932.jpg", "20251129_114957.jpg", "20251129_115010.jpg", "20251129_115017.jpg", "20251129_115123.jpg", "20251129_115140.jpg", "20251129_115201.jpg", "20251129_115218.jpg", "20251129_115223.jpg", "20251129_115251.jpg", "20251129_115305.jpg", "20251129_115313.jpg", "20251129_115322.jpg", "20251129_115331.jpg", "20251129_115426.jpg", "20251129_115503.jpg", "20251129_115534.jpg", "20251129_115633.jpg", "20251129_115719.jpg", "20251129_115734.jpg", "20251129_115754.jpg", "20251129_115815.jpg", "20251129_115822.jpg", "20251129_115827.jpg", "20251129_115900.jpg", "20251129_115923.jpg", "20251129_120045.jpg", "20251129_120152.jpg", "20251129_120217.jpg", "20251129_120227.jpg", "20251129_120259.jpg", "20251129_120417.jpg", "20251129_120512.jpg", "20251129_120518.jpg", "20251129_120531.jpg", "20251129_120544.jpg", "20251129_120552.jpg", "20251129_120745.jpg", "20251129_120748.jpg", "20251129_120803.jpg", "20251129_120820.jpg", "20251129_120831.jpg", "20251129_120901.jpg", "20251129_120915.jpg", "20251129_120956.jpg", "20251129_121017.jpg", "20251129_121030.jpg", "20251129_121051.jpg", "20251129_121134.jpg", "20251129_130238.jpg", "20251129_130306.jpg", "20251129_130402.jpg", "20251129_130457.jpg", "20251129_130522.jpg", "20251129_130644.jpg", "20251129_130715.jpg", "20251129_130723.jpg", "20251129_130733.jpg", "20251129_130802.jpg", "20251129_130807.jpg", "20251129_130817.jpg", "20251129_130850.jpg", "20251129_131040.jpg", "20251129_131046.jpg", "20251129_131056.jpg", "20251129_131125.jpg", "20251129_131202.jpg", "20251129_131320.jpg", "20251129_131514.jpg", "20251129_131521.jpg", "20251129_131525.jpg", "20251129_131529.jpg", "20251129_131655.jpg", "20251129_131720.jpg", "20251129_131750.jpg", "20251129_131815.jpg", "20251129_131820.jpg", "20251129_131839.jpg", "20251129_131901.jpg", "20251129_131914.jpg", "20251129_131938.jpg", "20251129_132038.jpg", "20251129_132125.jpg", "20251129_132200.jpg", "20251129_132354.jpg", "20251129_132449.jpg", "20251129_132501.jpg", "20251129_132528.jpg", "20251129_132534.jpg", "20251129_132732.jpg", "20251129_132751.jpg", "20251129_132832.jpg", "20251129_132854.jpg", "20251129_132915.jpg", "20251129_132921.jpg", "20251129_132954.jpg", "20251129_133000.jpg", "20251129_133018.jpg", "20251129_133340.jpg", "20251129_133353.jpg", "20251129_133423.jpg", "20251129_133456.jpg", "20251129_133515.jpg", "20251129_133519.jpg", "20251129_133625.jpg", "20251129_133633.jpg", "20251129_133740.jpg", "20251129_133814.jpg", "20251129_133850.jpg", "20251129_133912.jpg", "20251129_133953.jpg", "20251129_133959.jpg", "20251129_134106.jpg", "20251129_134335.jpg", "20251129_134401.jpg", "20251129_134420.jpg", "20251129_134502.jpg", "20251129_134506.jpg", "20251129_134517.jpg", "20251129_134527.jpg", "20251129_134557.jpg", "20251129_134615.jpg", "20251129_134713.jpg", "20251129_134724.jpg", "20251129_135000.jpg", "20251129_135004.jpg", "20251129_135020.jpg", "20251129_135027.jpg", "20251129_135036.jpg", "20251129_135042.jpg", "20251129_135118.jpg", "20251129_135142.jpg", "20251129_135152.jpg", "20251129_135205.jpg", "20251129_135212.jpg", "20251129_135217.jpg", "20251129_135225.jpg", "20251129_135235.jpg", "20251129_135253.jpg", "20251129_135335.jpg", "20251129_135341.jpg", "20251129_135422.jpg", "20251129_135428.jpg", "20251129_135448.jpg", "20251129_135510.jpg", "20251129_135538.jpg", "20251129_135551.jpg", "20251129_135556.jpg", "20251129_135755.jpg", "20251129_135804.jpg", "20251129_135827.jpg", "20251129_135845.jpg", "20251129_140008.jpg", "20251129_140011.jpg", "20251129_140020.jpg", "20251129_140038.jpg", "20251129_140128.jpg", "20251129_140136.jpg", "20251129_140207.jpg", "20251129_140216.jpg", "20251129_140227.jpg", "20251129_140240.jpg", "20251129_140502.jpg", "20251129_140505.jpg", "20251129_140513.jpg", "20251129_140528.jpg", "20251129_140538.jpg", "20251129_140553.jpg", "20251129_140618.jpg", "20251129_140653.jpg", "20251129_140657.jpg", "20251129_140708.jpg", "20251129_140713.jpg", "20251129_140744.jpg", "20251129_140816.jpg", "20251129_140820.jpg", "20251129_140842.jpg", "20251129_140850.jpg", "20251129_140907.jpg", "20251129_140938.jpg"], "sana": ["Sana 1.jpg", "Sana 100.jpg", "Sana 103.jpg", "Sana 106.jpg", "Sana 107.jpg", "Sana 109.jpg", "Sana 110.jpg", "Sana 111.jpg", "Sana 114.jpg", "Sana 115.jpg", "Sana 118.jpg", "Sana 121.jpg", "Sana 124.jpg", "Sana 125.jpg", "Sana 127.jpg", "Sana 129.jpg", "Sana 131.jpg", "Sana 138.jpg", "Sana 14.jpg", "Sana 141.jpg", "Sana 143.jpg", "Sana 147.jpg", "Sana 148.jpg", "Sana 15.jpg", "Sana 154.jpg", "Sana 156.jpg", "Sana 163.jpg", "Sana 169.jpg", "Sana 173.jpg", "Sana 178.jpg", "Sana 18.jpg", "Sana 183.jpg", "Sana 188.jpg", "Sana 192.jpg", "Sana 194.jpg", "Sana 199.jpg", "Sana 20.jpg", "Sana 201.jpg", "Sana 203.jpg", "Sana 205.jpg", "Sana 206.jpg", "Sana 207.jpg", "Sana 208.jpg", "Sana 214.jpg", "Sana 221.jpg", "Sana 222.jpg", "Sana 223.jpg", "Sana 224.jpg", "Sana 225.jpg", "Sana 227.jpg", "Sana 237.jpg", "Sana 240.jpg", "Sana 241.jpg", "Sana 243.jpg", "Sana 244.jpg", "Sana 245.jpg", "Sana 249.jpg", "Sana 251.jpg", "Sana 253.jpg", "Sana 254.jpg", "Sana 255.jpg", "Sana 258.jpg", "Sana 259.jpg", "Sana 260.jpg", "Sana 261.jpg", "Sana 264.jpg", "Sana 267.jpg", "Sana 268.jpg", "Sana 269.jpg", "Sana 27.jpg", "Sana 272.jpg", "Sana 276.jpg", "Sana 278.jpg", "Sana 288.jpg", "Sana 29.jpg", "Sana 293.jpg", "Sana 294.jpg", "Sana 295.jpg", "Sana 307.jpg", "Sana 311.jpg", "Sana 313.jpg", "Sana 32.jpg", "Sana 323.jpg", "Sana 332.jpg", "Sana 339.jpg", "Sana 34.jpg", "Sana 340.jpg", "Sana 345.jpg", "Sana 346.jpg", "Sana 347.jpg", "Sana 349.jpg", "Sana 35.jpg", "Sana 350.jpg", "Sana 351.jpg", "Sana 36.jpg", "Sana 361.jpg", "Sana 369.jpg", "Sana 37.jpg", "Sana 374.jpg", "Sana 375.jpg", "Sana 377.jpg", "Sana 380.jpg", "Sana 383.jpg", "Sana 384.jpg", "Sana 386.jpg", "Sana 387.jpg", "Sana 388.jpg", "Sana 389.jpg", "Sana 39.jpg", "Sana 392.jpg", "Sana 393.jpg", "Sana 395.jpg", "Sana 399.jpg", "Sana 40.jpg", "Sana 400.jpg", "Sana 401.jpg", "Sana 408.jpg", "Sana 410.jpg", "Sana 411.jpg", "Sana 413.jpg", "Sana 414.jpg", "Sana 419.jpg", "Sana 421.jpg", "Sana 423.jpg", "Sana 425.jpg", "Sana 426.jpg", "Sana 427.jpg", "Sana 43.jpg", "Sana 432.jpg", "Sana 436.jpg", "Sana 438.jpg", "Sana 442.jpg", "Sana 443.jpg", "Sana 446.jpg", "Sana 453.jpg", "Sana 455.jpg", "Sana 457.jpg", "Sana 46.jpg", "Sana 462.jpg", "Sana 464.jpg", "Sana 466.jpg", "Sana 469.jpg", "Sana 473.jpg", "Sana 476.jpg", "Sana 477.jpg", "Sana 48.jpg", "Sana 481.jpg", "Sana 486.jpg", "Sana 488.jpg", "Sana 489.jpg", "Sana 492.jpg", "Sana 496.jpg", "Sana 498.jpg", "Sana 500.jpg", "Sana 507.jpg", "Sana 509.jpg", "Sana 51.jpg", "Sana 511.jpg", "Sana 512.jpg", "Sana 513.jpg", "Sana 514.jpg", "Sana 515.jpg", "Sana 519.jpg", "Sana 52.jpg", "Sana 525.jpg", "Sana 530.jpg", "Sana 531.jpg", "Sana 532.jpg", "Sana 533.jpg", "Sana 536.jpg", "Sana 537.jpg", "Sana 539.jpg", "Sana 548.jpg", "Sana 549.jpg", "Sana 55.jpg", "Sana 551.jpg", "Sana 553.jpg", "Sana 555.jpg", "Sana 556.jpg", "Sana 557.jpg", "Sana 560.jpg", "Sana 565.jpg", "Sana 569.jpg", "Sana 57.jpg", "Sana 572.jpg", "Sana 575.jpg", "Sana 576.jpg", "Sana 577.jpg", "Sana 580.jpg", "Sana 581.jpg", "Sana 582.jpg", "Sana 583.jpg", "Sana 585.jpg", "Sana 586.jpg", "Sana 588.jpg", "Sana 59.jpg", "Sana 591.jpg", "Sana 598.jpg", "Sana 6.jpg", "Sana 60.jpg", "Sana 601.jpg", "Sana 602.jpg", "Sana 608.jpg", "Sana 610.jpg", "Sana 614.jpg", "Sana 617.jpg", "Sana 619.jpg", "Sana 621.jpg", "Sana 622.jpg", "Sana 624.jpg", "Sana 626.jpg", "Sana 629.jpg", "Sana 63.jpg", "Sana 634.jpg", "Sana 637.jpg", "Sana 638.jpg", "Sana 64.jpg", "Sana 643.jpg", "Sana 647.jpg", "Sana 651.jpg", "Sana 653.jpg", "Sana 655.jpg", "Sana 656.jpg", "Sana 657.jpg", "Sana 663.jpg", "Sana 667.jpg", "Sana 668.jpg", "Sana 67.jpg", "Sana 671.jpg", "Sana 672.jpg", "Sana 674.jpg", "Sana 677.jpg", "Sana 682.jpg", "Sana 685.jpg", "Sana 69.jpg", "Sana 691.jpg", "Sana 694.jpg", "Sana 696.jpg", "Sana 7.jpg", "Sana 70.jpg", "Sana 700.jpg", "Sana 701.jpg", "Sana 705.jpg", "Sana 708.jpg", "Sana 715.jpg", "Sana 717.jpg", "Sana 720.jpg", "Sana 723.jpg", "Sana 725.jpg", "Sana 728.jpg", "Sana 73.jpg", "Sana 731.jpg", "Sana 737.jpg", "Sana 739.jpg", "Sana 74.jpg", "Sana 741.jpg", "Sana 743.jpg", "Sana 75.jpg", "Sana 750.jpg", "Sana 752.jpg", "Sana 755.jpg", "Sana 757.jpg", "Sana 762.jpg", "Sana 766.jpg", "Sana 771.jpg", "Sana 775.jpg", "Sana 776.jpg", "Sana 777.jpg", "Sana 779.jpg", "Sana 78.jpg", "Sana 786.jpg", "Sana 788.jpg", "Sana 790.jpg", "Sana 794.jpg", "Sana 795.jpg", "Sana 80.jpg", "Sana 803.jpg", "Sana 808.jpg", "Sana 809.jpg", "Sana 81.jpg", "Sana 813.jpg", "Sana 814.jpg", "Sana 816.jpg", "Sana 817.jpg", "Sana 818.jpg", "Sana 819.jpg", "Sana 820.jpg", "Sana 822.jpg", "Sana 824.jpg", "Sana 827.jpg", "Sana 830.jpg", "Sana 839.jpg", "Sana 84.jpg", "Sana 841.jpg", "Sana 842.jpg", "Sana 843.jpg", "Sana 844.jpg", "Sana 845.jpg", "Sana 846.jpg", "Sana 847.jpg", "Sana 853.jpg", "Sana 857.jpg", "Sana 86.jpg", "Sana 860.jpg", "Sana 861.jpg", "Sana 863.jpg", "Sana 87.jpg", "Sana 873.jpg", "Sana 875.jpg", "Sana 877.jpg", "Sana 886.jpg", "Sana 89.jpg", "Sana 890.jpg", "Sana 892.jpg", "Sana 900.jpg", "Sana 902.jpg", "Sana 903.jpg", "Sana 905.jpg", "Sana 906.jpg", "Sana 91.jpg", "Sana 912.jpg", "Sana 913.jpg", "Sana 917.jpg", "Sana 919.jpg", "Sana 921.jpg", "Sana 928.jpg", "Sana 931.jpg", "Sana 932.jpg", "Sana 934.jpg", "Sana 935.jpg", "Sana 937.jpg", "Sana 938.jpg", "Sana 94.jpg", "Sana 940.jpg", "Sana 943.jpg", "Sana 951.jpg", "Sana 953.jpg", "Sana 956.jpg", "Sana 958.jpg", "Sana 960.jpg", "Sana 966.jpg", "Sana 968.jpg", "Sana 972.jpg", "Sana 976.jpg", "Sana 979.jpg", "Sana 983.jpg", "Sana 99.jpg", "Sana 991.jpg", "Sana 993.jpg"]}, "validation": {"ojo_de_gallo": ["20251129_092009.jpg", "20251129_092333.jpg", "20251129_092735.jpg", "20251129_092925.jpg", "20251129_093509.jpg", "20251129_093848.jpg", "20251129_094057.jpg", "20251129_094345.jpg", "20251129_094716.jpg", "20251129_095022.jpg", "20251129_095346.jpg", "20251129_095547.jpg", "20251129_095626.jpg", "20251129_095703.jpg", "20251129_095949.jpg", "20251129_100037.jpg", "20251129_100540.jpg", "20251129_100648.jpg", "20251129_100822.jpg", "20251129_101002.jpg", "20251129_101038.jpg", "20251129_101044.jpg", "20251129_101243.jpg", "20251129_101733.jpg", "20251129_102732.jpg", "20251129_102831.jpg", "20251129_103110.jpg", "20251129_103945.jpg", "20251129_104403.jpg", "20251129_104738.jpg", "20251129_104907.jpg", "20251129_104918.jpg", "20251129_105430.jpg", "20251129_105445.jpg", "20251129_114552.jpg", "20251129_114708.jpg", "20251129_114714.jpg", "20251129_114939.jpg", "20251129_114946.jpg", "20251129_115257.jpg", "20251129_115342.jpg", "20251129_115513.jpg", "20251129_115618.jpg", "20251129_115807.jpg", "20251129_120332.jpg", "20251129_120401.jpg", "20251129_120633.jpg", "20251129_120920.jpg", "20251129_130229.jpg", "20251129_130232.jpg", "20251129_130322.jpg", "20251129_130533.jpg", "20251129_130657.jpg", "20251129_131129.jpg", "20251129_131356.jpg", "20251129_131402.jpg", "20251129_131638.jpg", "20251129_132216.jpg", "20251129_132305.jpg", "20251129_132632.jpg", "20251129_132910.jpg", "20251129_133030.jpg", "20251129_133527.jpg", "20251129_133840.jpg", "20251129_133933.jpg", "20251129_133943.jpg", "20251129_134656.jpg", "20251129_134708.jpg", "20251129_134753.jpg", "20251129_135137.jpg", "20251129_135531.jpg", "20251129_135606.jpg", "20251129_135624.jpg", "20251129_135927.jpg", "20251129_140749.jpg"], "sana": ["Sana 135.jpg", "Sana 144.jpg", "Sana 155.jpg", "Sana 157.jpg", "Sana 161.jpg", "Sana 19.jpg", "Sana 197.jpg", "Sana 229.jpg", "Sana 239.jpg", "Sana 252.jpg", "Sana 257.jpg", "Sana 297.jpg", "Sana 300.jpg", "Sana 314.jpg", "Sana 316.jpg", "Sana 342.jpg", "Sana 352.jpg", "Sana 355.jpg", "Sana 356.jpg", "Sana 359.jpg", "Sana 379.jpg", "Sana 398.jpg", "Sana 403.jpg", "Sana 428.jpg", "Sana 435.jpg", "Sana 441.jpg", "Sana 467.jpg", "Sana 471.jpg", "Sana 472.jpg", "Sana 50.jpg", "Sana 524.jpg", "Sana 526.jpg", "Sana 528.jpg", "Sana 558.jpg", "Sana 567.jpg", "Sana 574.jpg", "Sana 592.jpg", "Sana 596.jpg", "Sana 611.jpg", "Sana 62.jpg", "Sana 632.jpg", "Sana 633.jpg", "Sana 652.jpg", "Sana 675.jpg", "Sana 684.jpg", "Sana 687.jpg", "Sana 693.jpg", "Sana 697.jpg", "Sana 703.jpg", "Sana 718.jpg", "Sana 72.jpg", "Sana 721.jpg", "Sana 749.jpg", "Sana 765.jpg", "Sana 781.jpg", "Sana 805.jpg", "Sana 826.jpg", "Sana 828.jpg", "Sana 856.jpg", "Sana 866.jpg", "Sana 876.jpg", "Sana 879.jpg", "Sana 882.jpg", "Sana 897.jpg", "Sana 910.jpg", "Sana 936.jpg", "Sana 939.jpg", "Sana 950.jpg", "Sana 957.jpg", "Sana 962.jpg", "Sana 974.jpg", "Sana 980.jpg", "Sana 987.jpg", "Sana 994.jpg"]}, "test": {"ojo_de_gallo": ["20251129_091427.jpg", "20251129_092633.jpg", "20251129_092724.jpg", "20251129_093405.jpg", "20251129_094358.jpg", "20251129_094634.jpg", "20251129_094823.jpg", "20251129_094932.jpg", "20251129_095649.jpg", "20251129_095825.jpg", "20251129_095836.jpg", "20251129_100024.jpg", "20251129_100529.jpg", "20251129_101314.jpg", "20251129_101826.jpg", "20251129_101959.jpg", "20251129_102055.jpg", "20251129_102103.jpg", "20251129_102113.jpg", "20251129_102445.jpg", "20251129_102558.jpg", "20251129_102920.jpg", "20251129_103018.jpg", "20251129_103228.jpg", "20251129_103610.jpg", "20251129_103921.jpg", "20251129_104056.jpg", "20251129_104456.jpg", "20251129_104554.jpg", "20251129_104647.jpg", "20251129_104818.jpg", "20251129_104945.jpg", "20251129_105017.jpg", "20251129_105036.jpg", "20251129_115401.jpg", "20251129_115438.jpg", "20251129_115623.jpg", "20251129_115857.jpg", "20251129_120211.jpg", "20251129_120253.jpg", "20251129_120538.jpg", "20251129_120659.jpg", "20251129_120717.jpg", "20251129_120953.jpg", "20251129_121103.jpg", "20251129_130241.jpg", "20251129_130419.jpg", "20251129_131111.jpg", "20251129_131229.jpg", "20251129_131241.jpg", "20251129_131303.jpg", "20251129_131351.jpg", "20251129_131423.jpg", "20251129_131620.jpg", "20251129_131753.jpg", "20251129_131923.jpg", "20251129_132248.jpg", "20251129_132523.jpg", "20251129_132727.jpg", "20251129_132740.jpg", "20251129_132859.jpg", "20251129_133824.jpg", "20251129_134344.jpg", "20251129_134917.jpg", "20251129_135125.jpg", "20251129_135900.jpg", "20251129_140025.jpg", "20251129_140043.jpg", "20251129_140236.jpg", "20251129_140508.jpg", "20251129_140516.jpg", "20251129_140753.jpg", "20251129_140759.jpg", "20251129_140802.jpg", "20251129_140813.jpg"], "sana": ["Sana 104.jpg", "Sana 119.jpg", "Sana 12.jpg", "Sana 122.jpg", "Sana 123.jpg", "Sana 133.jpg", "Sana 177.jpg", "Sana 187.jpg", "Sana 191.jpg", "Sana 193.jpg", "Sana 198.jpg", "Sana 204.jpg", "Sana 215.jpg", "Sana 23.jpg", "Sana 248.jpg", "Sana 250.jpg", "Sana 275.jpg", "Sana 279.jpg", "Sana 308.jpg", "Sana 309.jpg", "Sana 31.jpg", "Sana 312.jpg", "Sana 320.jpg", "Sana 341.jpg", "Sana 354.jpg", "Sana 360.jpg", "Sana 365.jpg", "Sana 382.jpg", "Sana 417.jpg", "Sana 420.jpg", "Sana 422.jpg", "Sana 431.jpg", "Sana 433.jpg", "Sana 45.jpg", "Sana 452.jpg", "Sana 490.jpg", "Sana 494.jpg", "Sana 516.jpg", "Sana 527.jpg", "Sana 571.jpg", "Sana 594.jpg", "Sana 600.jpg", "Sana 603.jpg", "Sana 615.jpg", "Sana 627.jpg", "Sana 635.jpg", "Sana 636.jpg", "Sana 649.jpg", "Sana 650.jpg", "Sana 660.jpg", "Sana 670.jpg", "Sana 68.jpg", "Sana 689.jpg", "Sana 711.jpg", "Sana 730.jpg", "Sana 732.jpg", "Sana 735.jpg", "Sana 745.jpg", "Sana 761.jpg", "Sana 77.jpg", "Sana 774.jpg", "Sana 79.jpg", "Sana 792.jpg", "Sana 832.jpg", "Sana 837.jpg", "Sana 838.jpg", "Sana 850.jpg", "Sana 869.jpg", "Sana 894.jpg", "Sana 914.jpg", "Sana 933.jpg", "Sana 959.jpg", "Sana 965.jpg", "Sana 97.jpg", "Sana 977.jpg", "Sana 990.jpg"]}}''')
POS = {}
for part in ["train", "validation", "test"]:
    rutas = [os.path.join(RUTAS[c], f) for c in ["ojo_de_gallo", "sana"] for f in PARTICION[part][c]]
    faltan = [r for r in rutas if not os.path.isfile(r)]
    assert not faltan, f"Faltan {len(faltan)} imágenes, por ejemplo {faltan[:3]}"
    POS[("mendeley", part)] = leer_varias(rutas, f"Mendeley {part}")

rutas_bracol = listar(RUTAS["bracol"])
print("BRACOL disponibles:", len(rutas_bracol))
div_b = dividir(len(rutas_bracol))
for part, ii in div_b.items():
    POS[("bracol", part)] = leer_varias([rutas_bracol[i] for i in ii], f"BRACOL {part}")

rutas_pg = {c: listar(os.path.join(RUTAS["pg26038"], c)) for c in ["healthy", "ojo", "roya"]}
print({c: len(v) for c, v in rutas_pg.items()})
POS[("pg26038", "test")] = leer_varias(sum(rutas_pg.values(), []), "PG 26038 (solo prueba)")

pd.DataFrame([{"fuente": f, "particion": p, "n": len(X)} for (f, p), X in POS.items()]).to_csv(
    os.path.join(SAL, "tablas", "conteo_positivos.csv"), index=False)

Mendeley train: 699 imágenes leídas de 699 en 187 s
Mendeley validation: 149 imágenes leídas de 149 en 41 s
Mendeley test: 151 imágenes leídas de 151 en 42 s
BRACOL disponibles: 1402
BRACOL train: 980 imágenes leídas de 981 en 50 s
BRACOL validation: 210 imágenes leídas de 210 en 7 s
BRACOL test: 211 imágenes leídas de 211 en 7 s
{'healthy': 20, 'ojo': 40, 'roya': 40}
PG 26038 (solo prueba): 100 imágenes leídas de 100 en 9 s


## 3. Imágenes que no son hojas de café (negativos)
1. Las mismas tres fuentes públicas de la Fase 2, 400 imágenes de cada una y con la misma partición 70/15/15:
   manos (`rock_paper_scissors`), objetos y escenas (`imagenette/160px`) y hojas de otros cultivos (`plant_village`).
2. **Nuevas: imágenes sin contenido** (300): colores lisos, degradados, ruido, imágenes muy borrosas y casi negras o
   casi blancas. Partición 70/15/15.
3. **Nuevas: 25 de las 50 fotos propias** se usan para entrenar y validar; las otras 25 quedan **solo para prueba**.

In [6]:
def cargar_negativos_tfds(nombre, split, n):
    if MOCK:
        rng = np.random.RandomState(abs(hash(nombre)) % 2**31)
        return rng.uniform(0, 255, (n, TAM, TAM, 3)).astype("float32"), {"nombre": nombre, "n_usadas": n}
    import tensorflow_datasets as tfds
    ds, info = tfds.load(nombre, split=split, as_supervised=True, with_info=True, shuffle_files=False)
    ds = ds.shuffle(1000, seed=SEMILLA).take(n)
    ims = [tf.image.resize(tf.cast(im, tf.float32), (TAM, TAM), method="bilinear").numpy() for im, _ in ds]
    ims = [np.repeat(i, 3, -1) if i.shape[-1] == 1 else i for i in ims]
    return np.array(ims, dtype="float32"), {"nombre": nombre, "homepage": info.homepage, "n_usadas": len(ims)}

FUENTES_TFDS = [("rock_paper_scissors", "train", "manos"), ("imagenette/160px", "train", "objetos y escenas"),
                ("plant_village", "train", "hojas de otros cultivos")]
NEG = {}
CITAS = []
for nombre, split, desc in FUENTES_TFDS:
    X_n, cita = cargar_negativos_tfds(nombre, split, N_NEG_TFDS)
    cita["descripcion"] = desc; CITAS.append(cita)
    idx = np.random.RandomState(SEMILLA).permutation(len(X_n))     # igual que en la Fase 2
    a, b = int(0.7 * len(idx)), int(0.85 * len(idx))
    for part, ii in [("train", idx[:a]), ("validation", idx[a:b]), ("test", idx[b:])]:
        NEG[(desc, part)] = X_n[ii]
    print(nombre, len(X_n))

def imagenes_sin_contenido(n, semilla):
    """Imágenes artificiales que no muestran ninguna hoja."""
    rng = np.random.RandomState(semilla)
    out, tipo = [], []
    imagenette = NEG[("objetos y escenas", "train")]
    for k in range(n):
        t = k % 6
        if t == 0:      # color liso
            im = np.ones((TAM, TAM, 3), "float32") * rng.uniform(0, 255, 3)
        elif t == 1:    # degradado
            g = np.linspace(0, 1, TAM, dtype="float32")[:, None, None] if rng.rand() < .5 else np.linspace(0, 1, TAM, dtype="float32")[None, :, None]
            c1, c2 = rng.uniform(0, 255, 3), rng.uniform(0, 255, 3)
            im = np.broadcast_to(c1 + (c2 - c1) * g, (TAM, TAM, 3)).astype("float32")
        elif t == 2:    # ruido sobre un color
            im = np.clip(rng.uniform(0, 255, 3) + rng.normal(0, rng.uniform(3, 40), (TAM, TAM, 3)), 0, 255)
        elif t == 3:    # casi negra o casi blanca (lente tapado, sobreexposición)
            base = rng.uniform(0, 25) if rng.rand() < .5 else rng.uniform(230, 255)
            im = np.clip(base + rng.normal(0, 3, (TAM, TAM, 3)), 0, 255)
        else:           # foto real muy borrosa
            src = imagenette[rng.randint(len(imagenette))].astype("uint8")
            im = np.asarray(Image.fromarray(src).filter(ImageFilter.GaussianBlur(rng.uniform(12, 30))), "float32")
        out.append(im.astype("float32")); tipo.append(["liso", "degradado", "ruido", "casi_negra_o_blanca", "borrosa", "borrosa"][t])
    return np.stack(out), tipo

X_sint, tipo_sint = imagenes_sin_contenido(300, SEMILLA)
for part, ii in dividir(len(X_sint)).items():
    NEG[("sin contenido", part)] = X_sint[ii]

rutas_np = listar(RUTAS["negativos_propios"])
print("Fotos propias:", len(rutas_np))
X_np = leer_varias(rutas_np, "Fotos propias")
idx_np = np.random.RandomState(SEMILLA).permutation(len(rutas_np))
mitad = len(idx_np) // 2
np_entreno, np_prueba = idx_np[:mitad], idx_np[mitad:]
n_val_np = max(1, len(np_entreno) // 5)
NEG[("fotos propias", "train")] = X_np[np_entreno[n_val_np:]]
NEG[("fotos propias", "validation")] = X_np[np_entreno[:n_val_np]]
NEG[("fotos propias", "test")] = X_np[np_prueba]
pd.DataFrame({"imagen": [os.path.basename(r) for r in rutas_np],
              "uso": ["prueba" if i in set(np_prueba) else ("validacion" if i in set(np_entreno[:n_val_np]) else "entrenamiento")
                      for i in range(len(rutas_np))]}).to_csv(os.path.join(SAL, "tablas", "fotos_propias_particion.csv"), index=False)

# Conjunto fijo de control: exactamente las imágenes con las que se encontró el problema
CONTROL_LISAS = np.stack([np.full((TAM, TAM, 3), v, "float32") for v in (0.0, 128.0, 255.0)])

conteo = [{"fuente": f, "particion": p, "n": len(X), "clase": "no es hoja"} for (f, p), X in NEG.items()]
conteo += [{"fuente": f, "particion": p, "n": len(X), "clase": "hoja de café"} for (f, p), X in POS.items()]
df_conteo = pd.DataFrame(conteo); df_conteo.to_csv(os.path.join(SAL, "tablas", "conteo_conjuntos.csv"), index=False)
print(df_conteo.pivot_table(index=["clase", "fuente"], columns="particion", values="n", aggfunc="sum").fillna(0).astype(int))
guardar_json(CITAS, "fuentes_negativos_citas.json")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/rock_paper_scissors/incomplete.HBRPOI_3.0.0/rock_paper_scissors-train.tfre…

Generating test examples...: 0 examples [00:00, ? examples/s]

... (16 outputs left out)

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.FNO6B9_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
plant_village 400
Fotos propias: 50
Fotos propias: 50 imágenes leídas de 50 en 5 s
particion                             test  train  validation
clase        fuente                                          
hoja de café bracol                    211    980         210
             mendeley                  151    699         149
             pg26038                   100      0           0
no es hoja   fotos propias              25     20           5
             hojas de otros cultivos    60    280          60
             manos                      60    280          60
             objetos y escenas          60    280          60
             sin contenido              45    210          45


In [7]:
   !pip install -q importlib_resources

## 4. Entrenamiento del verificador v2
Misma receta de la Fase 2: MobileNetV3Small con la base congelada, aumento de datos, promedio global, dropout 0,2 y una
neurona sigmoide. Adam 1e-3, 15 épocas, semilla 42. Como hay más hojas que negativos, se usan pesos por clase.

In [8]:
def juntar(part):
    Xp = np.concatenate([X for (f, p), X in POS.items() if p == part and f != "pg26038"])
    Xn = np.concatenate([X for (f, p), X in NEG.items() if p == part])
    X = np.concatenate([Xp, Xn]); y = np.concatenate([np.ones(len(Xp)), np.zeros(len(Xn))]).astype("float32")
    return X, y

Xtr, ytr = juntar("train"); Xva, yva = juntar("validation")
print("Entrenamiento:", Xtr.shape, "hojas:", int(ytr.sum()), "| validación:", Xva.shape, "hojas:", int(yva.sum()))
pesos = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * (ytr == 1).sum())}
print("Pesos por clase:", pesos)

fijar_semillas()
aumento = keras.Sequential([keras.layers.RandomFlip("horizontal"), keras.layers.RandomRotation(0.1),
                            keras.layers.RandomZoom(0.1)], name="data_augmentation")
base_v2 = keras.applications.MobileNetV3Small(input_shape=(TAM, TAM, 3), include_top=False,
                                              weights=None if MOCK else "imagenet", include_preprocessing=True)
base_v2.trainable = False
e = keras.Input((TAM, TAM, 3), name="imagen")
z = base_v2(aumento(e), training=False)
z = keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(z)
z = keras.layers.Dropout(0.2, name="dropout")(z)
z = keras.layers.Dense(1, activation="sigmoid", name="es_hoja_de_cafe")(z)
verificador_v2 = keras.Model(e, z, name="verificador_hoja_de_cafe_v2")
verificador_v2.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                       metrics=["accuracy", keras.metrics.AUC(name="auc")])
hist = verificador_v2.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCAS, batch_size=32,
                          class_weight=pesos, shuffle=True, verbose=2)
verificador_v2.save(os.path.join(SAL, "modelos", "verificador_hoja_de_cafe_v2.keras"))
guardar_json({k: [float(v) for v in vs] for k, vs in hist.history.items()}, "verificador_v2_historial.json")
del Xtr

Entrenamiento: (2749, 224, 224, 3) hojas: 1679 | validación: (589, 224, 224, 3) hojas: 359
Pesos por clase: {0: np.float64(1.2845794392523364), 1: np.float64(0.8186420488385944)}
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
86/86 - 28s - 325ms/step - accuracy: 0.7887 - auc: 0.8763 - loss: 0.4548 - val_accuracy: 0.9423 - val_auc: 0.9949 - val_loss: 0.2249
Epoch 2/15
86/86 - 4s - 46ms/step - accuracy: 0.9625 - auc: 0.9982 - loss: 0.1751 - val_accuracy: 0.9830 - val_auc: 0.9996 - val_loss: 0.1322
Epoch 3/15
86/86 - 5s - 57ms/step - accuracy: 0.9854 - auc: 0.9998 - loss: 0.1029 - val_accuracy: 0.9966 - val_auc: 1.0000 - val_loss: 0.0876
Epoch 4/15
86/86 - 4s - 49ms/step - accuracy: 0.9902 - auc: 0.9999 - loss: 0.0718 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0684
Epoch 5/15
86/86 - 5s - 61ms/step - accuracy: 0.9945 - auc: 1.0000 - loss: 0.0550 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0551
Epoch 6/15
86/86 - 4s - 50ms/step - accuracy: 0.9971 - auc: 

## 5. Comparación del verificador v1 (Fase 2) con el v2
Para cada conjunto de prueba se reporta la tasa de aceptación (hojas de café) o de rechazo (lo que no es hoja), con su
intervalo de confianza del 95 % de Wilson. Las imágenes de prueba no se usaron para entrenar ni para elegir el modelo.
Nota: las hojas de BRACOL y las imágenes sin contenido son nuevas para el v1, pero el v2 se entrenó con otras imágenes
de esos mismos tipos; por eso, en esos dos conjuntos la comparación favorece al v2 y se interpreta con cautela.
Las fotos propias de prueba, PG 26038 y las imágenes lisas de control son las comparaciones más justas.

In [10]:
def modelo_inferencia(m):
    """Mismo modelo sin las capas de aumento de datos."""
    ent = keras.Input((TAM, TAM, 3), name="imagen"); x = ent
    for capa in m.layers[1:]:
        if capa.name == "data_augmentation":
            continue
        x = capa(x, training=False)
    return keras.Model(ent, x)

v2_inf = modelo_inferencia(verificador_v2)
MODELOS = {"v2": v2_inf}
class ModeloTFLite:
    """El verificador v1 se evalúa con el mismo archivo .tflite que usaba la aplicación."""
    def __init__(self, ruta):
        self.it = tf.lite.Interpreter(model_path=ruta); self.it.allocate_tensors()
        self.i = self.it.get_input_details()[0]; self.o = self.it.get_output_details()[0]
    def predict(self, X, batch_size=None, verbose=0):
        s = []
        for k in range(len(X)):
            self.it.set_tensor(self.i["index"], X[k:k + 1]); self.it.invoke()
            s.append(float(self.it.get_tensor(self.o["index"]).ravel()[0]))
        return np.array(s)

RUTA_V1 = os.path.join(BASE, "fase2_resultados", "android", "modelo_verificador.tflite")
if os.path.isfile(RUTA_V1):
    MODELOS["v1"] = ModeloTFLite(RUTA_V1)
else:
    print("AVISO: no se encontró el verificador v1; solo se evalúa el v2")

PRUEBAS = [("Hojas Mendeley (prueba)", POS[("mendeley", "test")], 1),
           ("Hojas BRACOL (prueba)", POS[("bracol", "test")], 1),
           ("Hojas PG 26038 (externas)", POS[("pg26038", "test")], 1),
           ("Manos (prueba)", NEG[("manos", "test")], 0),
           ("Objetos y escenas (prueba)", NEG[("objetos y escenas", "test")], 0),
           ("Hojas de otros cultivos (prueba)", NEG[("hojas de otros cultivos", "test")], 0),
           ("Imágenes sin contenido (prueba)", NEG[("sin contenido", "test")], 0),
           ("Fotos propias (25 de prueba)", NEG[("fotos propias", "test")], 0),
           ("Control: gris, negra y blanca", CONTROL_LISAS, 0)]

def salida(m, X):
    return m.predict(X, batch_size=32, verbose=0).ravel() if len(X) else np.zeros(0)

filas, por_imagen = [], []
for nombre_m, m in MODELOS.items():
    for nombre, X, clase in PRUEBAS:
        p = salida(m, X); acepta = p >= UMBRAL_VERIFICADOR
        k = int(acepta.sum()) if clase == 1 else int((~acepta).sum())
        lo, hi = wilson(k, len(p))
        filas.append({"verificador": nombre_m, "conjunto": nombre, "n": len(p),
                      "medida": "aceptadas" if clase == 1 else "rechazadas", "aciertos": k,
                      "tasa": k / len(p) if len(p) else float("nan"), "IC95_inf": lo, "IC95_sup": hi})
        if nombre.startswith("Control"):
            print(nombre_m, "control gris/negra/blanca → P(hoja) =", np.round(p, 3))
df_cmp = pd.DataFrame(filas)
df_cmp.to_csv(os.path.join(SAL, "tablas", "comparacion_verificador_v1_v2.csv"), index=False)
tabla = df_cmp.pivot_table(index="conjunto", columns="verificador", values="tasa", sort=False)
print(tabla.round(3).to_string())

nombres_np = [os.path.basename(rutas_np[i]) for i in np_prueba]
fp = {"imagen": nombres_np}
for nombre_m, m in MODELOS.items():
    fp[f"p_hoja_{nombre_m}"] = salida(m, NEG[("fotos propias", "test")])
pd.DataFrame(fp).to_csv(os.path.join(SAL, "tablas", "fotos_propias_prueba_por_imagen.csv"), index=False)

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


v2 control gris/negra/blanca → P(hoja) = [0.022 0.017 0.155]
v1 control gris/negra/blanca → P(hoja) = [0.849 0.939 0.958]
verificador                         v2     v1
conjunto                                     
Hojas Mendeley (prueba)           1.00  0.815
Hojas BRACOL (prueba)             1.00  0.408
Hojas PG 26038 (externas)         0.91  0.660
Manos (prueba)                    1.00  1.000
Objetos y escenas (prueba)        1.00  1.000
Hojas de otros cultivos (prueba)  1.00  1.000
Imágenes sin contenido (prueba)   1.00  0.222
Fotos propias (25 de prueba)      1.00  1.000
Control: gris, negra y blanca     1.00  0.000


## 6. Filtro de calidad de la aplicación
Antes de los modelos, la aplicación calcula la desviación estándar de los valores de la imagen de 224 × 224
(todos los píxeles y canales). El umbral se fija con datos: **la mitad de la menor desviación observada en todas las
hojas de café reales** (Mendeley, BRACOL y PG 26038). Así, ninguna hoja real del estudio queda por debajo del umbral.

In [11]:
def desviacion(X):
    return X.reshape(len(X), -1).std(axis=1) if len(X) else np.zeros(0)

d_hojas = np.concatenate([desviacion(X) for X in POS.values()])
UMBRAL_DESVIACION = float(np.floor(d_hojas.min() / 2 * 10) / 10)
filtro = {"desviacion_hojas_min": float(d_hojas.min()), "desviacion_hojas_p1": float(np.percentile(d_hojas, 1)),
          "desviacion_hojas_mediana": float(np.median(d_hojas)), "umbral_desviacion": UMBRAL_DESVIACION,
          "n_hojas": int(len(d_hojas))}
for nombre, X, clase in PRUEBAS:
    if clase == 0:
        filtro[f"rechazadas_por_filtro::{nombre}"] = float((desviacion(X) < UMBRAL_DESVIACION).mean())
guardar_json(filtro, "filtro_calidad.json")
print(json.dumps(filtro, indent=2, ensure_ascii=False))

{
  "desviacion_hojas_min": 31.693931579589844,
  "desviacion_hojas_p1": 44.4549560546875,
  "desviacion_hojas_mediana": 63.664588928222656,
  "umbral_desviacion": 15.800000190734863,
  "n_hojas": 2500,
  "rechazadas_por_filtro::Manos (prueba)": 0.0,
  "rechazadas_por_filtro::Objetos y escenas (prueba)": 0.0,
  "rechazadas_por_filtro::Hojas de otros cultivos (prueba)": 0.0,
  "rechazadas_por_filtro::Imágenes sin contenido (prueba)": 0.1111111111111111,
  "rechazadas_por_filtro::Fotos propias (25 de prueba)": 0.0,
  "rechazadas_por_filtro::Control: gris, negra y blanca": 1.0
}


## 7. Conversión a TensorFlow Lite
Se convierten las variantes float32, float16 y cuantización dinámica. Se elige la más liviana que coincide con el
modelo de Keras en al menos el 99 % de las decisiones sobre todos los conjuntos de prueba.

In [12]:
try:
    from ai_edge_litert.interpreter import Interpreter
except Exception:
    Interpreter = tf.lite.Interpreter

def convertir(m, variante):
    carpeta = "/content/sm_v2" if not MOCK else "/tmp/sm_v2"
    try:
        m.export(carpeta, verbose=False)
    except TypeError:
        m.export(carpeta)
    c = tf.lite.TFLiteConverter.from_saved_model(carpeta)
    if variante in ("float16", "dinamica"):
        c.optimizations = [tf.lite.Optimize.DEFAULT]
    if variante == "float16":
        c.target_spec.supported_types = [tf.float16]
    return c.convert()

def correr(bytes_modelo, X):
    it = Interpreter(model_content=bytes_modelo); it.allocate_tensors()
    i = it.get_input_details()[0]; o = it.get_output_details()[0]
    s = []
    for k in range(len(X)):
        it.set_tensor(i["index"], X[k:k + 1]); it.invoke(); s.append(float(it.get_tensor(o["index"]).ravel()[0]))
    return np.array(s)

X_todas = np.concatenate([X for _, X, _ in PRUEBAS])
ref = salida(v2_inf, X_todas)
filas_t, binarios = [], {}
for variante in ["float32", "float16", "dinamica"]:
    b = convertir(v2_inf, variante); binarios[variante] = b
    s = correr(b, X_todas)
    filas_t.append({"variante": variante, "tamano_MB": len(b) / 1e6,
                    "acuerdo_decision": float(np.mean((s >= UMBRAL_VERIFICADOR) == (ref >= UMBRAL_VERIFICADOR))),
                    "dif_max_probabilidad": float(np.max(np.abs(s - ref)))})
df_t = pd.DataFrame(filas_t); print(df_t.to_string(index=False))
df_t.to_csv(os.path.join(SAL, "tablas", "verificador_v2_variantes_tflite.csv"), index=False)
ok = df_t[df_t.acuerdo_decision >= 0.99].sort_values("tamano_MB")
ELEGIDA = ok.iloc[0]["variante"] if len(ok) else "float32"
ruta_app = os.path.join(SAL, "android", "modelo_verificador.tflite")
open(ruta_app, "wb").write(binarios[ELEGIDA])
print("Variante elegida:", ELEGIDA, "→", ruta_app, f"({len(binarios[ELEGIDA]) / 1e6:.2f} MB)")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use 

variante  tamano_MB  acuerdo_decision  dif_max_probabilidad
 float32   3.754212           1.00000              0.000012
 float16   1.943624           1.00000              0.024222
dinamica   1.131384           0.99021              0.574789
Variante elegida: dinamica → /content/drive/MyDrive/tesis_ojo_de_gallo/fase2b_verificador/android/modelo_verificador.tflite (1.13 MB)


## 8. Resumen para la tesis

In [13]:
resumen = {"versiones": VERSIONES, "epocas": EPOCAS, "semilla": SEMILLA, "pesos_clase": pesos,
           "conteo": conteo, "comparacion": filas, "filtro_calidad": filtro,
           "tflite": filas_t, "variante_elegida": ELEGIDA}
resumen["imagenes_danadas"] = DANADAS
guardar_json(resumen, "resumen_fase2b.json")
print("Guardado en", SAL)
print(tabla.round(3).to_string())

Guardado en /content/drive/MyDrive/tesis_ojo_de_gallo/fase2b_verificador
verificador                         v2     v1
conjunto                                     
Hojas Mendeley (prueba)           1.00  0.815
Hojas BRACOL (prueba)             1.00  0.408
Hojas PG 26038 (externas)         0.91  0.660
Manos (prueba)                    1.00  1.000
Objetos y escenas (prueba)        1.00  1.000
Hojas de otros cultivos (prueba)  1.00  1.000
Imágenes sin contenido (prueba)   1.00  0.222
Fotos propias (25 de prueba)      1.00  1.000
Control: gris, negra y blanca     1.00  0.000


In [14]:
# 1) Guardar la variante float16 del verificador v2 para la app
open(os.path.join(SAL, "android", "modelo_verificador.tflite"), "wb").write(binarios["float16"])
print("Verificador v2 float16 guardado:", round(len(binarios["float16"]) / 1e6, 2), "MB")

from sklearn.metrics import roc_auc_score
diag_b = open(os.path.join(BASE, "fase2_resultados", "android", "modelo_diagnostico.tflite"), "rb").read()
ver_b = binarios["float16"]

def leer_sin_exif(ruta):
    with Image.open(ruta) as im:
        arr = np.asarray(im.convert("RGB"), dtype=np.float32)
    return tf.image.resize(arr[None], (TAM, TAM), method="bilinear").numpy()[0]

def orientacion(ruta):
    with Image.open(ruta) as im:
        return im.getexif().get(274, 1)

def regla(ph, po):
    return np.where(ph < 0.5, "imagen_no_valida", np.where(po >= 0.7, "compatible",
           np.where(po <= 0.3, "sin_sintomas", "no_concluyente")))

# 2) ¿Afecta la orientación EXIF al diagnóstico? (prueba de Mendeley, 151 imágenes)
rt = [os.path.join(RUTAS[c], f) for c in ["ojo_de_gallo", "sana"] for f in PARTICION["test"][c]]
yt = np.array([1] * len(PARTICION["test"]["ojo_de_gallo"]) + [0] * len(PARTICION["test"]["sana"]))
ori = [orientacion(r) for r in rt]
print("Orientación EXIF en prueba:", pd.Series(ori).value_counts().to_dict())
exif = {}
for nombre, lector in [("con_EXIF_(como_la_app)", leer), ("sin_EXIF_(como_Fase2)", leer_sin_exif)]:
    X = np.stack([lector(r) for r in rt]).astype("float32")
    po = 1 - correr(diag_b, X); ph = correr(ver_b, X)
    vp = int(((po >= 0.5) & (yt == 1)).sum()); vn = int(((po < 0.5) & (yt == 0)).sum())
    exif[nombre] = {"exactitud": (vp + vn) / len(yt), "sensibilidad": vp / yt.sum(), "especificidad": vn / (yt == 0).sum(),
                    "auc": roc_auc_score(yt, po), "verificador_v2_acepta": float((ph >= 0.5).mean()),
                    "regla_app": pd.Series(regla(ph, po)).value_counts().to_dict()}
guardar_json(exif, "diagnostico_efecto_exif.json")
print(json.dumps(exif, indent=2, default=float))

# 3) Roya (500) y PG 26038 (validación externa exploratoria), leídas como en la app
def evaluar(nombre, rutas):
    ok, X = [], []
    for r in rutas:
        try: X.append(leer(r)); ok.append(r)
        except Exception as e: print("dañada:", r, e)
    X = np.stack(X).astype("float32")
    ph = correr(ver_b, X); po = 1 - correr(diag_b, X)
    return pd.DataFrame({"conjunto": nombre, "imagen": [os.path.basename(r) for r in ok],
                         "p_hoja": ph, "p_ojo": po, "regla_app": regla(ph, po)})

roya = evaluar("Roya Mendeley", listar(os.path.join(DS, "Roya")))
pg = pd.concat([evaluar(f"PG26038 {c}", rutas_pg[c]) for c in ["ojo", "healthy", "roya"]])
todo = pd.concat([roya, pg])
todo.to_csv(os.path.join(SAL, "tablas", "diagnostico_roya_pg26038_por_imagen.csv"), index=False)
print(todo.groupby("conjunto")["regla_app"].value_counts().unstack(fill_value=0))

ojo = pg[pg.conjunto == "PG26038 ojo"]; san = pg[pg.conjunto == "PG26038 healthy"]
vp = int((ojo.p_ojo >= 0.5).sum()); vn = int((san.p_ojo < 0.5).sum())
auc = roc_auc_score(np.r_[np.ones(len(ojo)), np.zeros(len(san))], np.r_[ojo.p_ojo, san.p_ojo])
def por_hoja(df): return df.assign(hoja=df.imagen.str.split("__").str[0]).groupby("hoja").p_ojo.mean()
ho, hs = por_hoja(ojo), por_hoja(san)
vph, vnh = int((ho >= 0.5).sum()), int((hs < 0.5).sum())
ext = {"imagenes_ojo": len(ojo), "imagenes_sanas": len(san),
       "sensibilidad": vp / len(ojo), "sens_IC95": wilson(vp, len(ojo)),
       "especificidad": vn / len(san), "esp_IC95": wilson(vn, len(san)), "auc": auc,
       "hojas_ojo": len(ho), "hojas_sanas": len(hs),
       "sens_por_hoja": vph / len(ho), "sens_hoja_IC95": wilson(vph, len(ho)),
       "esp_por_hoja": vnh / len(hs), "esp_hoja_IC95": wilson(vnh, len(hs))}
guardar_json(ext, "validacion_externa_exploratoria_pg26038.json")
print(json.dumps(ext, indent=2, default=float))

Verificador v2 float16 guardado: 1.94 MB
Orientación EXIF en prueba: {6: 145, 1: 6}


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


{
  "con_EXIF_(como_la_app)": {
    "exactitud": 0.9933774834437086,
    "sensibilidad": 1.0,
    "especificidad": 0.9868421052631579,
    "auc": 1.0,
    "verificador_v2_acepta": 1.0,
    "regla_app": {
      "sin_sintomas": 75,
      "compatible": 73,
      "no_concluyente": 3
    }
  },
  "sin_EXIF_(como_Fase2)": {
    "exactitud": 1.0,
    "sensibilidad": 1.0,
    "especificidad": 1.0,
    "auc": 1.0,
    "verificador_v2_acepta": 0.9867549668874173,
    "regla_app": {
      "compatible": 75,
      "sin_sintomas": 74,
      "imagen_no_valida": 2
    }
  }
}
dañada: /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Roya/Roya 752.jpg broken data stream when reading image file


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


regla_app        compatible  imagen_no_valida  no_concluyente  sin_sintomas
conjunto                                                                   
PG26038 healthy           0                 0               4            16
PG26038 ojo              15                 9               9             7
PG26038 roya              2                 0              22            16
Roya Mendeley           159                 5              99           236
{
  "imagenes_ojo": 40,
  "imagenes_sanas": 20,
  "sensibilidad": 0.65,
  "sens_IC95": [
    0.49505880837257693,
    0.7786547112682372
  ],
  "especificidad": 0.95,
  "esp_IC95": [
    0.763868806553258,
    0.9911185511992047
  ],
  "auc": 0.89625,
  "hojas_ojo": 20,
  "hojas_sanas": 10,
  "sens_por_hoja": 0.65,
  "sens_hoja_IC95": [
    0.43285427668523624,
    0.818808175898918
  ],
  "esp_por_hoja": 1.0,
  "esp_hoja_IC95": [
    0.7224672001371107,
    0.9999999999999999
  ]
}
